# 6. 동적 생산일정 의사결정 지원 시스템 개발 로드맵

> **Set A 기반 사전 Cycle 예측과 7개 Scheduling Result 비교를 분리한 뒤, Result Viewer에서 동적 생산일정 의사결정 지원 시스템으로 확장하는 로드맵**

현재 프로젝트에서 검증한 것은 Block Cycle 예측과 이미 계산된 Result 비교다. 새로운 일정을 자동 생성하거나 현장 실적을 반영한 동적 최적화를 구현한 것은 아니며, 이후 단계는 추가 데이터와 Scheduling Engine이 확보될 경우의 개발 방향이다.


## 6.1 출발점과 개발 목표

현재 프로젝트에서 확보한 기반은 다음과 같다.

- Block / Position 후보 / Initial / Result의 입력층·결과층 구분
- Block 고유 특성으로 사전 `Block Position Cycle`을 예측하는 Set A 모델
- 7개 방법의 Position 배정, Planned Schedule, Makespan과 Delay 비교
- Result를 공간·시간 관점으로 확인하는 Streamlit Viewer

Streamlit Viewer는 결과 전달뿐 아니라 분석 가정을 검증하는 역할도 했다. 같은 Block의 선택 Position이 Scheduling Method에 따라 달라지는 것을 시각적으로 확인하면서, 선택 Position을 사전 예측변수로 취급한 문제를 발견했다.

목표 시스템은 두 역할을 분리한다.

```text
Cycle Prediction Model: Block 특성 → 기준 처리기간 추정
Scheduling Engine: Block / Yard / Initial / Cycle → 실행가능 일정 후보
Decision Support: 후보별 Position / Time / Delay / Cost 비교
```

머신러닝이 Position을 임의로 정하거나 하나의 일정을 자동 확정하는 것이 아니라, 생산관리자가 여러 후보를 비교하도록 돕는 구조를 목표로 한다.


## 6.2 단계별 개발 로드맵

| Phase | 상태 | 핵심 목표 |
|---:|---|---|
| 1 | 완료 | Set A Cycle 예측과 7개 Result 비교 구조 확립 |
| 2 | 구현 중 | 7개 Scheduling Result 공간·시간 Viewer |
| 3 | 중기 | 지연위험·이상배정·Position 혼잡 Analyzer |
| 4 | 장기 | 실제 실적을 반영한 Cycle 재예측과 Dynamic Rescheduling |
| 5 | 장기 | Calendar-it을 통한 개인·팀 업무일정 연결 |
| 6 | 장기 | ERP 관점의 자원·자재·원가 연결 |
| 7 | 최종 확장 | MES 실적과 재예측·재스케줄링 순환구조 |

현재 바로 검증할 범위는 **Phase 2 Result Viewer**다. Viewer는 새 일정을 생성하지 않고 게시자가 제공한 7개 결과를 정확히 재현·비교한다.


## 6.3 Phase 2 — Scheduling Result Viewer

### 6.3.1 목표

DDQN, EDDQN과 5개 Heuristic Result를 동일한 공간·시간 인터페이스에서 비교한다.

```text
Scheduling Method 선택
        ↓
Result의 Block–Position 배치 표시
        ↓
Planned Start / Finish 시간 필터
        ↓
Position·Block 상세정보와 Gantt Chart
        ↓
Makespan / Delay / Position 사용 비교
```

### 6.3.2 입력 데이터

- Block / Position / Initial 입력 Table
- 7개 Scheduling Result Table
- 3번 노트북의 Set A Cycle 예측모델
- 4번 노트북의 Position·Schedule·Delay 요약 결과

Set A 예측은 아직 배정되지 않은 Block의 기준 처리기간 추정에 사용한다. Viewer의 Position·시간·Delay는 회귀모델이 새로 생성하지 않고 원본 Result에서 읽는다.

### 6.3.3 핵심 기능

- Scheduling Method 전환
- 동일 Block의 방법별 Position 변화 확인
- 시점별 Position 점유와 Gantt Chart
- Makespan, Total Delay, Delayed Blocks, Max Delay 비교
- Set A Cycle 예측과 Scheduling Result를 화면상에서 명확히 구분


## 6.4 Phase 3 — Scheduling Analyzer

Viewer가 안정적으로 동작하면 다음 기능을 추가한다.

- **지연위험 분류**: Block별 Delay 위험 Low / Medium / High
- **군집**: 물리·일정 특성이 유사한 Block 그룹
- **이상탐지**: 비정상 Cycle, Delay, 과도한 Position 집중
- **민감도 분석**: Makespan·Delay·Utilization·Cost 가중치에 따른 순위 변화
- **모델 진단**: Set A에서 큰 잔차가 발생한 Block과 누락 조건 탐색

이 단계에서도 선택 Position을 Cycle의 원인으로 단정하지 않는다. 목표는 어떤 Block과 조건에서 특정 일정이 불리해지는지 설명하는 것이다.


## 6.5 Phase 4 — Dynamic Scheduling System

실제 착수·완료시점과 현장상태를 확보하면 정적인 결과 비교에서 동적 일정관리로 확장한다.

```text
Block 특성 → 사전 Cycle 예측
                  ↓
        초기 Scheduling Engine
                  ↓
        실행 일정과 현장 실적
                  ↓
        잔여기간·지연위험 재예측
                  ↓
        Dynamic Rescheduling
                  ↓
 Makespan / Delay / Cost / Utilization 후보 비교
                  ↓
        생산관리자 최종 선택
```

Position과 Initial은 이 단계에서 Scheduling Engine의 제약조건으로 사용한다. 스케줄링 이후 확보된 실적정보는 다음 재예측 시점의 입력으로 사용할 수 있지만, 최초 사전 Cycle 모델과 구분해 관리한다.


## 6.6 Phase 5–7 — 업무·자원·현장실적 연계

### 6.6.1 Phase 5 — Calendar-it 연계

생산 Schedule을 담당자·협력사의 준비, 착수, 검사 및 완료 Task로 변환하여 개인·팀 일정과 연결한다. 일정 변경 시 관련 담당자가 영향을 받는 작업을 함께 확인하는 협업구조를 목표로 한다.

### 6.6.2 Phase 6 — ERP 개념 연계

- 자재 입고와 구매 일정
- Man-hour와 작업조 부하
- 협력사 배정
- 인건비·설비비·물류비
- Delay에 따른 비용 변화
- 스케줄링 방법별 자원·원가 비교

이를 통해 현재 프로젝트에서 평가하지 못한 Cost와 자원 관점의 평가축을 추가한다.

### 6.6.3 Phase 7 — MES 연계

MES의 작업지시, 착수·완료시점, 생산량, Man-hour, 설비상태, 품질검사와 재작업 정보를 다시 예측·스케줄링 시스템으로 전달한다. 이 단계에서 계획과 실제 생산현장을 연결하는 피드백 구조가 완성된다.

## 6.7 목표 시스템의 피드백 구조

```text
Block Master ──→ Set A Cycle Prediction
                         ↓
Position / Initial ─→ Scheduling Engine ←─ ERP 자원·자재·원가
                         ↓
              실행가능 Schedule 후보
                         ↓
              3D/Gantt Result Viewer
                         ↓
                생산관리자 의사결정
                         ↓
         MES 착수·완료·Man-hour·품질 실적
                         ↓
             재예측 및 Dynamic Rescheduling
                         └──────────────↺
```

핵심은 **예측모델과 Scheduling Engine의 역할을 분리**한 뒤 계획 → 실행 → 실적 → 재예측 → 재스케줄링을 연결하는 것이다.


## 6.8 바로 시작할 작업

- [x] Set A Cycle 예측과 Scheduling Result 비교 역할 분리
- [x] 7개 Result 공통 Long Format 및 주요 지표 검증
- [x] Scheduling Method 선택과 Gantt Chart 구현
- [ ] 시점별 Position 점유 상태와 Yard 개념도 연결
- [ ] Makespan·Delay·Position 사용 비교화면 정리
- [ ] Set A 예측화면과 Result Viewer의 용어·동선 분리
- [ ] 실제 Yard 좌표·실적 Cycle이 없다는 제한 표시
- [ ] 7개 방법 전환 시 행 수와 집계값 자동 검증

기능을 늘리기 전에 원본 Result와 화면의 Block 수, Position, 날짜와 Delay가 정확히 일치하는지 검증한다.


## 6.9 Phase 2 완료 기준

- 7개 Scheduling Method를 같은 화면에서 전환할 수 있다.
- 선택한 시점의 Block–Position 점유를 확인할 수 있다.
- Block/Position 선택 시 원본 Planned Schedule과 Delay를 확인할 수 있다.
- Gantt Chart와 Makespan·Delay 요약이 4번 노트북과 일치한다.
- Set A Cycle 예측과 Result 조회가 서로 다른 기능으로 표시된다.
- 실제 Yard 좌표를 사용하지 않은 개념도라는 제한이 표시된다.
- Cycle Stability 등 폐기한 지표가 핵심 성과로 노출되지 않는다.


## 6.10 최종 목표

> **Set A Cycle Prediction → Scheduling Engine → Result Viewer → Scheduling Analyzer → Dynamic Rescheduling → ERP/MES 연계 순으로 확장한다.**

궁극적으로는 머신러닝이 Position과 일정을 자동으로 결정하는 시스템이 아니라, 기준 처리기간을 추정하고 여러 실행가능 일정의 공간·시간·지연·비용 결과를 비교해 생산관리자의 판단을 지원하는 시스템을 목표로 한다.
